# 02 — EDA: Engagement Patterns

> **Notebook 02 of 7** | Learning Retention Analytics  
> Second exploratory analysis: how do students interact with the platform, and what do engagement patterns reveal?

## Purpose and Scope

This notebook is the **second of two EDA notebooks**. Notebook 01 profiled the *student base* (demographics, outcomes, course landscape). Here we shift to **behavioral engagement** — the clickstream patterns that reflect what students actually *do* on the platform.

**What this notebook covers:**
- Ghost students — enrollments with zero VLE activity
- Distribution of early engagement metrics (first 28 days)
- Dose-response relationship between engagement decile and outcome
- Daily engagement trajectory — when does the gap open?
- Engagement typology — binge vs. steady patterns
- Activity type diversity — what resources do students use?
- Course-level engagement profiles
- Engagement-outcome correlation summary

**What comes next:**
- **Notebook 03** (`03_bq1_dropout_timing.ipynb`): where and when do students drop out? (BQ1)

**Connection to business questions:**  
This EDA does not answer BQ1–BQ5 directly. Instead, it builds the **behavioral foundation** for BQ2 (early signals predicting dropout), BQ3 (demographics vs. behavior), and BQ5 (actionable interventions). Think of it as *mapping the behavioral landscape before testing specific hypotheses*.

> **Methodological transferability:** The engagement patterns explored here — ghost users, onboarding-window signals, binge vs. steady usage, activity diversity — are directly portable to SaaS retention, subscription churn, and fitness app engagement. The *domain* is education; the *analytical framework* is product analytics.

## Table of Contents

1. [Environment Setup](#1.-Environment-Setup)
2. [Ghost Students — Zero Engagement](#2.-Ghost-Students-—-Zero-Engagement)
3. [Early Engagement Distributions](#3.-Early-Engagement-Distributions)
4. [Engagement Deciles vs. Outcome](#4.-Engagement-Deciles-vs.-Outcome)
5. [Daily Engagement Trajectory](#5.-Daily-Engagement-Trajectory)
6. [Engagement Typology — Binge vs. Steady](#6.-Engagement-Typology-—-Binge-vs.-Steady)
7. [Activity Type Diversity](#7.-Activity-Type-Diversity)
8. [Course-Level Engagement Profiles](#8.-Course-Level-Engagement-Profiles)
9. [Engagement-Outcome Correlation Matrix](#9.-Engagement-Outcome-Correlation-Matrix)
10. [Key Takeaways and Next Steps](#10.-Key-Takeaways-and-Next-Steps)

---

**Prerequisites:**
- The ETL pipeline must have been run: `python -m run_pipeline`
- The DuckDB database at `data/db/oulad.duckdb` must contain all 5 analytical views

**Dataset:** Open University Learning Analytics Dataset (OULAD) — ~32K students, 7 courses, complete behavioral clickstream. License: CC-BY 4.0.

## 1. Environment Setup

We configure imports, visualization defaults, and reusable helper functions.

**Technical notes for readers:**
- Notebooks live in `notebooks/` but project modules are in `src/` at the project root. We add the project root to `sys.path` so that `from src.config import ...` works. The linter rule `E402` (imports not at top of file) is suppressed for notebooks in `pyproject.toml`.
- All database queries go through `src.db.connection.execute_query()` — the project's DB abstraction layer. This returns a `pandas.DataFrame` and ensures we never call `duckdb.connect()` directly (see [ADR-003](../docs/ADR.md)).
- Figures are saved to `reports/figures/` at 150 DPI. Since `nbstripout` removes notebook outputs before commit, the saved PNGs are the persistent visual record.

In [ ]:
# --- Path setup ---
# Notebooks live in notebooks/ but project modules are at the project root.
# Adding the root to sys.path lets us import from src/ as if running from root.
# We search upward for pyproject.toml instead of assuming cwd is always notebooks/,
# so the notebook works regardless of where the kernel is launched from
# (JupyterLab, VS Code, Cursor, repo root, etc.).
import sys
from pathlib import Path


def _find_project_root(start: Path) -> Path:
    """Walk upward from start until we find pyproject.toml (the repo root marker)."""
    for candidate in (start, *start.parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise RuntimeError(
        f"Could not locate project root: no pyproject.toml found in '{start}' "
        "or any parent directory. Run the notebook from within the repository."
    )


PROJECT_ROOT = _find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# --- Standard library ---
import logging
import warnings

# --- Third-party ---
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns

# --- Project modules ---
from src.config import FIGURES_DIR
from src.db.connection import execute_query

# --- Configuration ---
# Suppress noisy warnings in notebook output; errors still surface
warnings.filterwarnings('ignore', category=FutureWarning)
logging.basicConfig(level=logging.WARNING)

# --- Visualization defaults ---
# Consistent style across all project notebooks
sns.set_theme(style='whitegrid', font_scale=1.1)

# Semantic color palette: one color per outcome category
PALETTE_OUTCOME = {
    'Pass': '#4C72B0',        # blue — neutral positive
    'Distinction': '#55A868', # green — strong positive
    'Fail': '#C44E52',        # red — negative
    'Withdrawn': '#8172B3',   # purple — departed (distinct from failed)
}
# Outcome label constants — single source of truth for the binary labels
# used in SQL output mapping, palette keys, and plot iterations
LABEL_COMPLETED = 'Completed'
LABEL_NOT_COMPLETED = 'Not completed'
# Binary version: completed (1) vs not completed (0)
PALETTE_BINARY = {1: '#55A868', 0: '#C44E52'}
LABEL_BINARY = {1: LABEL_COMPLETED, 0: LABEL_NOT_COMPLETED}
# Label-keyed palette for seaborn when x-axis uses mapped string categories
PALETTE_BINARY_LABELS = {LABEL_COMPLETED: '#55A868', LABEL_NOT_COMPLETED: '#C44E52'}
# Sequential palette for heatmaps and continuous scales
PALETTE_SEQUENTIAL = 'YlOrRd'
# Shared axis labels — avoids cross-cell string literal duplication
LABEL_NUM_ENROLLMENTS = 'Number of enrollments'
LABEL_ACTIVE_DAYS = 'Active days (first 28)'
LABEL_COMPLETION_RATE = 'Completion rate (%)'

FIG_DPI = 150
FIG_SIZE = (10, 6)
FIG_SIZE_WIDE = (16, 5)

# Ensure figures output directory exists
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def save_fig(fig, name: str) -> None:
    """Save figure to reports/figures/ with consistent settings."""
    path = FIGURES_DIR / f'{name}.png'
    fig.savefig(path, dpi=FIG_DPI, bbox_inches='tight', facecolor='white')
    print(f'  Saved: {path}')


# --- Prerequisite check ---
# Verify the database is populated before proceeding.
# We check both engagement views since this notebook depends on them.
try:
    _check_daily = execute_query('SELECT COUNT(*) AS n FROM v_engagement_daily')
    _check_early = execute_query('SELECT COUNT(*) AS n FROM v_engagement_early')
    _check_student = execute_query('SELECT COUNT(*) AS n FROM v_student_enriched')
    _n_daily = _check_daily['n'].iloc[0]
    _n_early = _check_early['n'].iloc[0]
    _n_student = _check_student['n'].iloc[0]
    if _n_daily == 0 or _n_early == 0 or _n_student == 0:
        raise RuntimeError('One or more views are empty')
    print('Database OK')
    print(f'  v_engagement_daily:  {_n_daily:>12,} rows')
    print(f'  v_engagement_early:  {_n_early:>12,} rows')
    print(f'  v_student_enriched:  {_n_student:>12,} rows')
except Exception as exc:
    raise RuntimeError(
        'Cannot query engagement views. '
        "Run 'python -m run_pipeline' first to populate the database."
    ) from exc

## 2. Ghost Students — Zero Engagement

The most extreme behavioral signal is **no behavior at all**. A "ghost student" is an enrollment present in `v_student_enriched` but absent from `v_engagement_early` — meaning zero VLE clicks in the first 28 days.

**Why start here?** Ghost students establish the *floor* of engagement. Before analyzing how much active students click, we need to know how many students never clicked at all, and what happened to them.

**SaaS parallel:** Ghost students are the education equivalent of *dormant users* — people who signed up but never activated. In product analytics, activation rate is the first retention lever: you cannot retain users who never started using the product.

In [ ]:
# --- Ghost student identification ---
# v_engagement_early only includes enrollments with >= 1 VLE click in days 0-28.
# Enrollments absent from that view had zero activity — these are "ghost students".
# The LEFT JOIN + IS NULL pattern identifies them precisely per enrollment
# (a student can be a ghost in one course but active in another).
df_ghost_summary = execute_query('''
    SELECT
        COUNT(*) AS n_total,
        SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END) AS n_ghost,
        SUM(CASE WHEN ee.id_student IS NOT NULL THEN 1 ELSE 0 END) AS n_active
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
''')

n_total = df_ghost_summary['n_total'].iloc[0]
n_ghost = df_ghost_summary['n_ghost'].iloc[0]
n_active = df_ghost_summary['n_active'].iloc[0]

print('=== Ghost Student Overview ===\n')
print(f'  Total enrollments:       {n_total:>8,}')
print(f'  Active (>= 1 click):     {n_active:>8,} ({100 * n_active / n_total:.1f}%)')
print(f'  Ghost (zero activity):   {n_ghost:>8,} ({100 * n_ghost / n_total:.1f}%)')

# --- Completion rate comparison ---
df_ghost_outcome = execute_query('''
    SELECT
        CASE WHEN ee.id_student IS NULL THEN 'Ghost' ELSE 'Active' END AS segment,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY (ee.id_student IS NULL)
    ORDER BY segment
''')

print('\n=== Completion Rate: Ghost vs. Active ===\n')
print(df_ghost_outcome.to_string(index=False))

In [ ]:
# --- Ghost students by course ---
# Are ghost students concentrated in specific courses, or distributed evenly?
df_ghost_by_course = execute_query('''
    SELECT
        se.code_module,
        COUNT(*) AS n_enrolled,
        SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END) AS n_ghost,
        ROUND(
            100.0 * SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END)
            / COUNT(*), 1
        ) AS ghost_pct,
        ROUND(
            100.0 * SUM(CASE
                WHEN ee.id_student IS NOT NULL THEN se.completed ELSE 0
            END)
            / NULLIF(SUM(CASE WHEN ee.id_student IS NOT NULL THEN 1 ELSE 0 END), 0),
            1
        ) AS active_completion_pct,
        ROUND(
            100.0 * SUM(CASE
                WHEN ee.id_student IS NULL THEN se.completed ELSE 0
            END)
            / NULLIF(SUM(CASE WHEN ee.id_student IS NULL THEN 1 ELSE 0 END), 0),
            1
        ) AS ghost_completion_pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    GROUP BY se.code_module
    ORDER BY ghost_pct DESC
''')

# --- Visualization: 1x2 panel ---
# Left: ghost count per module. Right: completion rate comparison (ghost vs active).
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left panel: ghost student count by module
modules = df_ghost_by_course['code_module']
ax1.barh(modules, df_ghost_by_course['n_ghost'], color='#8172B3', edgecolor='white')
for i, (_, row) in enumerate(df_ghost_by_course.iterrows()):
    ax1.text(
        row['n_ghost'] + ax1.get_xlim()[1] * 0.01, i,
        f"{row['ghost_pct']:.1f}% of enrolled",
        va='center', fontsize=9, color='#333333',
    )
ax1.set_xlabel('Number of ghost enrollments')
ax1.set_title('Ghost Students by Module')
ax1.invert_yaxis()
sns.despine(ax=ax1)

# Right panel: completion rate — ghost vs active per module
x = np.arange(len(modules))
bar_width = 0.35
ax2.barh(x - bar_width / 2, df_ghost_by_course['active_completion_pct'],
         bar_width, label='Active', color='#55A868', edgecolor='white')
ax2.barh(x + bar_width / 2, df_ghost_by_course['ghost_completion_pct'],
         bar_width, label='Ghost', color='#C44E52', edgecolor='white')
ax2.set_yticks(x)
ax2.set_yticklabels(modules)
ax2.set_xlabel(LABEL_COMPLETION_RATE)
ax2.set_title('Completion Rate: Active vs. Ghost')
ax2.legend(loc='lower right')
ax2.invert_yaxis()
sns.despine(ax=ax2)

fig.suptitle('Ghost Students Across Modules', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '02_ghost_students_by_course')
plt.show()

In [ ]:
# --- 4-class outcome distribution for ghost students ---
# What happens to students who never engage? Are they all Withdrawn,
# or do some manage to Pass/Fail through other channels (e.g. exams only)?
df_ghost_4class = execute_query('''
    SELECT
        se.final_result,
        COUNT(*) AS n,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
    WHERE ee.id_student IS NULL
    GROUP BY se.final_result
    ORDER BY n DESC
''')

print('=== Ghost Student Outcome Distribution ===\n')
print(df_ghost_4class.to_string(index=False))

# --- Horizontal bar chart ---
fig, ax = plt.subplots(figsize=FIG_SIZE)
colors = [PALETTE_OUTCOME.get(r, '#999999') for r in df_ghost_4class['final_result']]
bars = ax.barh(df_ghost_4class['final_result'], df_ghost_4class['n'], color=colors,
               edgecolor='white')

for bar, (_, row) in zip(bars, df_ghost_4class.iterrows()):
    ax.text(
        bar.get_width() + ax.get_xlim()[1] * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{int(row['n']):,} ({row['pct']:.1f}%)",
        va='center', fontsize=11,
    )

ax.set_xlabel(LABEL_NUM_ENROLLMENTS)
ax.set_title('Outcome Distribution — Ghost Students Only (zero VLE activity)')
ax.invert_yaxis()
sns.despine(left=True)
fig.tight_layout()
save_fig(fig, '02_ghost_outcome_distribution')
plt.show()

> **Key finding:** Ghost students have a near-zero completion rate. The vast majority are **Withdrawn** — they enrolled, never interacted with the VLE, and eventually departed.
>
> - Ghost students are present across all modules, not concentrated in a single course.
> - The completion rate gap between active and ghost students is enormous — this is the clearest engagement-outcome signal in the dataset.
> - A small number of ghost students may still Pass or Fail (e.g., through in-person exams or banked credits), but they are a tiny minority.
>
> **Implication for BQ5:** Ghost students are the lowest-hanging fruit for intervention. They need *activation*, not academic support. In SaaS terms, this is the "onboarding gap" — the user signed up but never experienced the core product value.
>
> **From here on**, Sections 3–9 analyze only **active students** (those with at least one VLE click in the first 28 days), unless explicitly noted otherwise.

## 3. Early Engagement Distributions

For students who **did** engage (non-ghosts), what does their early activity look like? This section profiles the four key metrics from `v_engagement_early`:

| Metric | What it measures |
|--------|-----------------|
| `active_days_first_28` | Number of distinct days with at least one click (0–28) |
| `total_clicks_first_28` | Total VLE clicks across all resources |
| `avg_clicks_per_active_day` | Intensity: clicks per day when present |
| `last_active_day_in_window` | Last day of activity within the 28-day window |

We compare these distributions between **Completed** and **Not completed** enrollments to build visual intuition before the formal statistical tests in Notebook 04 (BQ2).

> **Note:** Ghost students (zero activity) are excluded from `v_engagement_early` by construction. The distributions here represent the *active* population only.

In [ ]:
# --- Summary statistics by outcome ---
# Mean vs median gap reveals skewness (common in clickstream data:
# a few power users generate disproportionate click volumes).
df_early_stats = execute_query('''
    SELECT
        CASE WHEN se.completed = 1 THEN 'Completed' ELSE 'Not completed' END AS outcome,
        COUNT(*) AS n,
        ROUND(AVG(ee.active_days_first_28), 1) AS mean_active_days,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.active_days_first_28), 1
        ) AS median_active_days,
        ROUND(AVG(ee.total_clicks_first_28), 0) AS mean_total_clicks,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.total_clicks_first_28), 0
        ) AS median_total_clicks,
        ROUND(AVG(ee.avg_clicks_per_active_day), 1) AS mean_clicks_per_day,
        ROUND(AVG(ee.last_active_day_in_window), 1) AS mean_last_active_day
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY se.completed
    ORDER BY se.completed DESC
''')

print('=== Early Engagement Summary (first 28 days, active students only) ===\n')
df_early_stats

In [ ]:
# --- 2x2 violin plot grid: all 4 early metrics split by outcome ---
# Violins show the full distribution shape — more informative than box plots
# for detecting bimodality or heavy tails in engagement data.
df_early = execute_query('''
    SELECT
        ee.active_days_first_28,
        ee.total_clicks_first_28,
        ee.avg_clicks_per_active_day,
        ee.last_active_day_in_window,
        se.completed
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
''')
df_early['outcome'] = df_early['completed'].map(LABEL_BINARY)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

metrics = [
    ('active_days_first_28', LABEL_ACTIVE_DAYS, axes[0, 0]),
    ('total_clicks_first_28', 'Total clicks (first 28)', axes[0, 1]),
    ('avg_clicks_per_active_day', 'Avg clicks per active day', axes[1, 0]),
    ('last_active_day_in_window', 'Last active day in window', axes[1, 1]),
]

for col, ylabel, ax in metrics:
    sns.violinplot(
        data=df_early, x='outcome', y=col,
        palette=PALETTE_BINARY_LABELS, inner='quartile', ax=ax,
    )
    ax.set_xlabel('')
    ax.set_ylabel(ylabel)
    sns.despine(ax=ax)

fig.suptitle(
    'Early Engagement Metrics by Completion Outcome (first 28 days)',
    fontsize=14, y=1.02,
)
fig.tight_layout()
save_fig(fig, '02_early_engagement_violins')
plt.show()

In [ ]:
# --- Overlapping histograms: active days and total clicks ---
# Histograms complement violins by showing raw frequency counts,
# making it easier to see where the bulk of enrollments falls.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=FIG_SIZE_WIDE)

# Left: active days
for outcome_val, label in LABEL_BINARY.items():
    subset = df_early[df_early['completed'] == outcome_val]
    ax1.hist(
        subset['active_days_first_28'], bins=28, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )
ax1.set_xlabel(LABEL_ACTIVE_DAYS)
ax1.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax1.set_title('Active Days Distribution by Outcome')
ax1.legend()
sns.despine(ax=ax1)

# Right: total clicks (cap at 99th percentile to avoid outlier compression)
p99 = df_early['total_clicks_first_28'].quantile(0.99)
for outcome_val, label in LABEL_BINARY.items():
    subset = df_early[df_early['completed'] == outcome_val]
    ax2.hist(
        subset['total_clicks_first_28'].clip(upper=p99), bins=50, alpha=0.6,
        label=label, color=PALETTE_BINARY[outcome_val],
    )
ax2.set_xlabel(f'Total clicks (first 28 days, capped at p99 = {p99:,.0f})')
ax2.set_ylabel(LABEL_NUM_ENROLLMENTS)
ax2.set_title('Total Clicks Distribution by Outcome')
ax2.legend()
sns.despine(ax=ax2)

fig.suptitle('Early Engagement Distributions', fontsize=14, y=1.02)
fig.tight_layout()
save_fig(fig, '02_early_engagement_histograms')
plt.show()

> **Interpretation:**
> - All four metrics show **visible separation** between completers and non-completers. Completers tend to have more active days, more total clicks, and their last active day is later in the 28-day window.
> - The **total clicks** distribution is heavily right-skewed — a few power users generate disproportionate click volumes. The mean is substantially higher than the median, confirming this skewness.
> - The **last active day** metric is particularly telling: non-completers' activity tends to trail off earlier in the window, while completers remain active closer to day 28.
> - The **active days** histogram shows that many non-completers are active for only 1–5 days out of 28 — brief engagement before disappearing.
>
> **Causation caveat:** Higher engagement is *associated with* completion, but we cannot say engagement *causes* completion from this data alone. Motivated students may both engage more and complete more — engagement could be a proxy for motivation, not a cause of success. Notebook 04 (BQ2) will quantify these associations with formal statistical tests and effect sizes.

## 4. Engagement Deciles vs. Outcome

Is there a **dose-response** relationship between engagement and completion? If more engagement monotonically predicts better outcomes, this strengthens the case for engagement-based interventions.

`v_engagement_early` includes an `engagement_decile_in_course` column: each student is ranked 1–10 within their own course-presentation using `NTILE(10)` over `total_clicks_first_28`. This normalization is critical — it ensures that decile 1 in a high-engagement course and decile 1 in a low-engagement course both mean "bottom 10% of *that* course."

In [ ]:
# --- Completion rate by engagement decile ---
# If the relationship is dose-response, we expect a monotonic increase
# from decile 1 (lowest engagement) to decile 10 (highest engagement).
df_decile = execute_query('''
    SELECT
        ee.engagement_decile_in_course AS decile,
        COUNT(*) AS n,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY ee.engagement_decile_in_course
    ORDER BY ee.engagement_decile_in_course
''')

print('=== Completion Rate by Engagement Decile ===\n')
print(df_decile.to_string(index=False))

# --- Bar chart with gradient coloring ---
# Color gradient from red (low decile) to green (high decile) reinforces
# the dose-response visual pattern and matches the binary palette semantics.
overall_rate = df_decile['completion_rate_pct'].mean()
n_deciles = len(df_decile)
gradient_colors = [
    plt.cm.RdYlGn(i / (n_deciles - 1)) for i in range(n_deciles)
]

fig, ax = plt.subplots(figsize=FIG_SIZE)
bars = ax.bar(
    df_decile['decile'], df_decile['completion_rate_pct'],
    color=gradient_colors, edgecolor='white',
)

# Annotate each bar with the exact percentage
for bar, (_, row) in zip(bars, df_decile.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
        f"{row['completion_rate_pct']:.1f}%",
        ha='center', fontsize=9, color='#333333',
    )

# Reference line: overall average (across active students only)
ax.axhline(y=overall_rate, color='gray', linestyle='--', linewidth=1,
           label=f'Average: {overall_rate:.1f}%')

ax.set_xlabel('Engagement decile (1 = lowest, 10 = highest)')
ax.set_ylabel(LABEL_COMPLETION_RATE)
ax.set_title('Completion Rate by Engagement Decile (within-course normalized)')
ax.set_xticks(range(1, 11))
ax.legend(loc='upper left')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_decile_completion_rate')
plt.show()

In [ ]:
# --- 4-class outcome breakdown by decile (100% stacked bar) ---
# Shows how the mix of Pass/Distinction/Fail/Withdrawn changes across deciles.
# Complements the binary view above with the full outcome granularity.
df_decile_4class = execute_query('''
    SELECT
        ee.engagement_decile_in_course AS decile,
        se.final_result,
        COUNT(*) AS n
    FROM v_engagement_early ee
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY ee.engagement_decile_in_course, se.final_result
    ORDER BY ee.engagement_decile_in_course, se.final_result
''')

# Pivot and normalize to 100%
df_pivot = df_decile_4class.pivot(
    index='decile', columns='final_result', values='n',
).fillna(0)
df_pct = df_pivot.div(df_pivot.sum(axis=1), axis=0) * 100

# Order columns: positive outcomes first, then negative
col_order = ['Distinction', 'Pass', 'Fail', 'Withdrawn']
df_pct = df_pct[[c for c in col_order if c in df_pct.columns]]

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_pct.plot.bar(
    stacked=True, ax=ax,
    color=[PALETTE_OUTCOME[c] for c in df_pct.columns],
    edgecolor='white', linewidth=0.5,
)
ax.set_xlabel('Engagement decile (1 = lowest, 10 = highest)')
ax.set_ylabel('Percentage of enrollments')
ax.set_title('Outcome Mix by Engagement Decile')
ax.legend(title='Outcome', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
sns.despine()
fig.tight_layout()
save_fig(fig, '02_decile_outcome_stacked')
plt.show()

> **Interpretation:**
> - The dose-response pattern is clear: completion rate increases monotonically (or near-monotonically) from the lowest to the highest engagement decile.
> - The bottom 2–3 deciles are an **extreme-risk segment** — their completion rates are far below average.
> - In the stacked bar chart, the Withdrawn segment (purple) shrinks dramatically as engagement increases, while Distinction (green) grows. This suggests that higher engagement is associated with both lower dropout and higher academic achievement.
> - Because deciles are **within-course normalized**, this pattern is robust to course-level differences in click volume. A student in decile 1 of any course is at high risk.
>
> **Actionable insight (BQ5 preview):** An engagement-based early warning system could flag students in the bottom 2–3 deciles within the first 28 days. In SaaS terms, this is equivalent to a *health score* that triggers proactive outreach for at-risk users.

## 5. Daily Engagement Trajectory

Sections 3–4 summarized engagement as a single number over 28 days. Here we add the **temporal dimension**: how does engagement evolve day by day?

The key question: **at what point do the trajectories of completers and non-completers diverge?** If the gap opens early (e.g., within the first week), the intervention window is narrow but actionable. If it opens gradually, interventions have more time but may need to be more persistent.

We use `v_engagement_daily`, which provides per-day granularity (total clicks, distinct resources, activity types) for every student-day with at least one click.

In [ ]:
# --- Average daily clicks by outcome group (days 0-28) ---
# We compute the mean clicks per day for each outcome group.
# Students who did not click on a given day are NOT in v_engagement_daily
# for that day — so this average is "clicks per active student on that day",
# not "clicks per enrolled student". The active student count plot below
# provides the complementary view.
df_trajectory = execute_query('''
    SELECT
        ed.date AS day,
        CASE WHEN se.completed = 1 THEN 'Completed' ELSE 'Not completed' END AS outcome,
        ROUND(AVG(ed.total_clicks), 2) AS avg_clicks,
        COUNT(*) AS n_active_enrollments
    FROM v_engagement_daily ed
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    WHERE ed.date BETWEEN 0 AND 28
    GROUP BY ed.date, se.completed
    ORDER BY ed.date, se.completed
''')

fig, ax = plt.subplots(figsize=FIG_SIZE)
for outcome in ['Completed', LABEL_NOT_COMPLETED]:
    subset = df_trajectory[df_trajectory['outcome'] == outcome]
    ax.plot(
        subset['day'], subset['avg_clicks'],
        label=outcome, color=PALETTE_BINARY_LABELS[outcome],
        linewidth=2,
    )

# Shade the area between the two lines to emphasize divergence
df_comp = df_trajectory[df_trajectory['outcome'] == 'Completed'].set_index('day')
df_notc = df_trajectory[df_trajectory['outcome'] == LABEL_NOT_COMPLETED].set_index('day')
common_days = df_comp.index.intersection(df_notc.index)
ax.fill_between(
    common_days,
    df_comp.loc[common_days, 'avg_clicks'],
    df_notc.loc[common_days, 'avg_clicks'],
    alpha=0.1, color='gray',
)

ax.set_xlabel('Day (relative to course start)')
ax.set_ylabel('Average clicks per active student')
ax.set_title('Daily Engagement Trajectory (first 28 days)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '02_daily_trajectory_first_28')
plt.show()

In [ ]:
# --- Active enrollment count per day by outcome ---
# This contextualizes the trajectory above: does the gap widen because
# non-completers click less, or because they stop clicking entirely?
# A declining line for non-completers means students are dropping out
# of activity — not just reducing their intensity.
fig, ax = plt.subplots(figsize=FIG_SIZE)
for outcome in ['Completed', LABEL_NOT_COMPLETED]:
    subset = df_trajectory[df_trajectory['outcome'] == outcome]
    ax.plot(
        subset['day'], subset['n_active_enrollments'],
        label=outcome, color=PALETTE_BINARY_LABELS[outcome],
        linewidth=2,
    )

ax.set_xlabel('Day (relative to course start)')
ax.set_ylabel('Number of active enrollments')
ax.set_title('Active Enrollments per Day (first 28 days)')
ax.legend()
sns.despine()
fig.tight_layout()
save_fig(fig, '02_daily_active_students_first_28')
plt.show()

> **Interpretation:**
> - The trajectory plot shows a **persistent gap** between completers and non-completers throughout the first 28 days. Completers consistently click more on days they are active.
> - The active enrollment count reveals a **dual signal**: non-completers not only click less when present, but also *stop showing up* earlier. The declining red line represents students who have effectively disengaged from the platform.
> - The combination of these two plots is powerful: the engagement gap is driven by both **intensity** (less clicking per visit) and **frequency** (fewer visits, earlier dropout from activity).
>
> **Intervention implication:** The divergence likely becomes visible within the first 7–10 days. This narrows the intervention window: by the time the first month is over, many non-completers have already stopped engaging. Early detection systems need to act within the first two weeks to have the greatest impact.

## 6. Engagement Typology — Binge vs. Steady

Not all engagement is created equal. Two students with 200 total clicks can have very different patterns:
- **Binge**: 200 clicks in 2 days (concentrated bursts, cramming behavior)
- **Steady**: 200 clicks spread across 14 days (consistent, distributed engagement)

The `active_days_first_28` and `total_clicks_first_28` dimensions capture this distinction. A scatter plot of these two variables, colored by outcome, reveals whether *consistency* or *volume* matters more for completion.

**SaaS parallel:** This maps to the difference between DAU (daily active users) and session depth. A user who logs in briefly every day is behaviorally different from one who has one marathon session per week — even if their total usage time is similar.

In [ ]:
# --- Scatter plot: active days vs total clicks, colored by outcome ---
# We reuse df_early (loaded in Section 3) which already contains all 4 metrics
# plus the outcome column.
# Cap total_clicks at the 99th percentile to prevent a few extreme outliers
# from compressing the visual range for the majority of students.
p99_clicks = df_early['total_clicks_first_28'].quantile(0.99)

fig, ax = plt.subplots(figsize=FIG_SIZE)
# Plot non-completers first (background) so completers are visible on top
for outcome_val in [0, 1]:
    label = LABEL_BINARY[outcome_val]
    subset = df_early[df_early['completed'] == outcome_val]
    ax.scatter(
        subset['active_days_first_28'],
        subset['total_clicks_first_28'].clip(upper=p99_clicks),
        alpha=0.15, s=10, label=label,
        color=PALETTE_BINARY[outcome_val],
    )

# Annotate quadrant labels to guide interpretation
ax.text(2, p99_clicks * 0.90, 'Binge\n(few days, many clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(22, p99_clicks * 0.90, 'Power User\n(many days, many clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(2, p99_clicks * 0.05, 'Minimal\n(few days, few clicks)',
        fontsize=10, color='#666666', style='italic')
ax.text(22, p99_clicks * 0.05, 'Steady\n(many days, moderate clicks)',
        fontsize=10, color='#666666', style='italic')

ax.set_xlabel('Active days (first 28)')
ax.set_ylabel(f'Total clicks (first 28 days, capped at p99 = {p99_clicks:,.0f})')
ax.set_title('Engagement Typology: Active Days vs. Total Clicks')
ax.legend(markerscale=5)
sns.despine()
fig.tight_layout()
save_fig(fig, '02_engagement_typology_scatter')
plt.show()

In [ ]:
# --- Typology heatmap: median-split quadrants → completion rate ---
# Split students into 4 quadrants using the median of active_days and
# the median of avg_clicks_per_active_day. This quantifies whether
# consistency (high active days) or intensity (high clicks per day) is
# more strongly associated with completion.
df_thresholds = execute_query('''
    SELECT
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY active_days_first_28)
            AS median_active_days,
        PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY avg_clicks_per_active_day)
            AS median_intensity
    FROM v_engagement_early
''')

med_days = df_thresholds['median_active_days'].iloc[0]
med_intensity = df_thresholds['median_intensity'].iloc[0]

print(f'Median active days: {med_days:.0f}')
print(f'Median clicks per active day: {med_intensity:.1f}')

# Classify each enrollment into a quadrant
df_early['frequency'] = np.where(
    df_early['active_days_first_28'] >= med_days, 'High frequency', 'Low frequency'
)
df_early['intensity'] = np.where(
    df_early['avg_clicks_per_active_day'] >= med_intensity, 'High intensity', 'Low intensity'
)

# Compute completion rate per quadrant
df_typology = (
    df_early.groupby(['frequency', 'intensity'])
    .agg(n=('completed', 'count'), completion_rate=('completed', 'mean'))
    .reset_index()
)
df_typology['completion_rate'] = (df_typology['completion_rate'] * 100).round(1)

# Pivot for heatmap
heatmap_data = df_typology.pivot(
    index='intensity', columns='frequency', values='completion_rate',
)
heatmap_counts = df_typology.pivot(
    index='intensity', columns='frequency', values='n',
)

# Reorder for intuitive reading: high on top, low on bottom
row_order = ['High intensity', 'Low intensity']
col_order = ['Low frequency', 'High frequency']
heatmap_data = heatmap_data.loc[row_order, col_order]
heatmap_counts = heatmap_counts.loc[row_order, col_order]

# Annotate with both completion rate and count
annot = heatmap_data.copy().astype(str)
for r in row_order:
    for c in col_order:
        rate = heatmap_data.loc[r, c]
        count = heatmap_counts.loc[r, c]
        annot.loc[r, c] = f'{rate:.1f}%\n(n={int(count):,})'

fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(
    heatmap_data, annot=annot, fmt='', cmap='RdYlGn',
    vmin=0, vmax=100, linewidths=1, linecolor='white',
    cbar_kws={'label': 'Completion rate (%)'}, ax=ax,
)
ax.set_title('Engagement Typology: Completion Rate by Quadrant\n'
             f'(split at median: {med_days:.0f} active days, '
             f'{med_intensity:.1f} clicks/day)')
ax.set_xlabel('')
ax.set_ylabel('')
fig.tight_layout()
save_fig(fig, '02_engagement_typology_heatmap')
plt.show()

> **Interpretation:**
> - The scatter plot shows that **completers cluster in the upper-right** (many active days, high total clicks), while non-completers dominate the lower-left (few days, few clicks).
> - The "binge" quadrant (few days, many clicks) is relatively sparse — most students with high click volumes also spread them across multiple days.
> - The 2×2 heatmap quantifies this: **high frequency is the stronger predictor**. Moving from low to high frequency (more active days) increases completion rate more than moving from low to high intensity (more clicks per session). Consistency beats cramming.
> - The "minimal" quadrant (low frequency, low intensity) has the lowest completion rate — these are students who barely engaged even when they did show up.
>
> **Implication for BQ5:** Interventions should prioritize *consistency* over *volume*. Encouraging students to log in regularly (even briefly) may be more effective than encouraging longer sessions. This is analogous to the "daily streak" mechanic used in consumer apps to build habit formation.

## 7. Activity Type Diversity

Sections 3–6 focused on *how much* students engage (clicks, days, intensity). This section asks *what* they engage with.

The OULAD VLE contains several activity types: content pages (`oucontent`), forums (`forumng`), quizzes, resources, homepage, subpages, and more. The `vle` table maps each resource (`id_site`) to its `activity_type`.

**Diversity hypothesis:** Students who interact with a wider variety of resource types may be more deeply engaged with the course — accessing not just core content but also forums, quizzes, and supplementary materials. If diversity correlates with completion, it suggests that *breadth* of engagement matters alongside *volume*.

In [ ]:
# --- Activity type popularity: which resource types get the most clicks? ---
# Filtering to first 28 days to match the engagement window used throughout.
df_activity_pop = execute_query('''
    SELECT
        v.activity_type,
        SUM(sv.sum_click) AS total_clicks,
        ROUND(100.0 * SUM(sv.sum_click) / SUM(SUM(sv.sum_click)) OVER (), 1)
            AS pct_of_clicks
    FROM studentVle sv
    JOIN vle v
        ON sv.id_site = v.id_site
        AND sv.code_module = v.code_module
        AND sv.code_presentation = v.code_presentation
    WHERE sv.date BETWEEN 0 AND 28
    GROUP BY v.activity_type
    ORDER BY total_clicks DESC
''')

print('=== Activity Type Popularity (first 28 days) ===\n')
print(df_activity_pop.to_string(index=False))

In [ ]:
# --- Activity type usage by outcome ---
# Which activity types show the largest gap between completers and non-completers?
# We compute average clicks per enrollment (not per click) to normalize for
# the different group sizes.
df_activity_outcome = execute_query('''
    SELECT
        v.activity_type,
        CASE WHEN se.completed = 1 THEN 'Completed' ELSE 'Not completed' END AS outcome,
        ROUND(
            SUM(sv.sum_click) * 1.0
            / COUNT(DISTINCT se.id_student || '-' || se.code_module || '-' || se.code_presentation),
            1
        ) AS avg_clicks_per_enrollment
    FROM studentVle sv
    JOIN vle v
        ON sv.id_site = v.id_site
        AND sv.code_module = v.code_module
        AND sv.code_presentation = v.code_presentation
    JOIN v_student_enriched se
        ON sv.id_student = se.id_student
        AND sv.code_module = se.code_module
        AND sv.code_presentation = se.code_presentation
    WHERE sv.date BETWEEN 0 AND 28
    GROUP BY v.activity_type, se.completed
    ORDER BY v.activity_type, se.completed
''')

# Pivot for grouped bar chart
df_act_pivot = df_activity_outcome.pivot(
    index='activity_type', columns='outcome', values='avg_clicks_per_enrollment',
).fillna(0)

# Sort by the gap between Completed and Not completed (largest gap first)
df_act_pivot['_gap'] = (
    df_act_pivot.get('Completed', 0) - df_act_pivot.get('Not completed', 0)
)
df_act_pivot = df_act_pivot.sort_values('_gap', ascending=True)
df_act_pivot = df_act_pivot.drop(columns='_gap')

fig, ax = plt.subplots(figsize=FIG_SIZE)
df_act_pivot.plot.barh(
    ax=ax,
    color=[PALETTE_BINARY_LABELS.get(c, '#999999') for c in df_act_pivot.columns],
    edgecolor='white',
)
ax.set_xlabel('Average clicks per enrollment (first 28 days)')
ax.set_ylabel('')
ax.set_title('Activity Type Usage by Outcome')
ax.legend(title='Outcome')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_activity_type_by_outcome')
plt.show()

In [ ]:
# --- Activity type diversity per student vs completion ---
# Count how many distinct activity types each student used in first 28 days,
# then compute completion rate per diversity level.
df_diversity = execute_query('''
    SELECT
        sub.n_activity_types,
        COUNT(*) AS n_enrollments,
        ROUND(100.0 * SUM(se.completed) / COUNT(*), 1) AS completion_rate_pct
    FROM (
        SELECT
            sv.id_student,
            sv.code_module,
            sv.code_presentation,
            COUNT(DISTINCT v.activity_type) AS n_activity_types
        FROM studentVle sv
        JOIN vle v
            ON sv.id_site = v.id_site
            AND sv.code_module = v.code_module
            AND sv.code_presentation = v.code_presentation
        WHERE sv.date BETWEEN 0 AND 28
        GROUP BY sv.id_student, sv.code_module, sv.code_presentation
    ) sub
    JOIN v_student_enriched se
        USING (id_student, code_module, code_presentation)
    GROUP BY sub.n_activity_types
    ORDER BY sub.n_activity_types
''')

print('=== Activity Type Diversity vs. Completion Rate ===\n')
print(df_diversity.to_string(index=False))

> **Interpretation:**
> - A few activity types dominate the click volume — `oucontent` (course content pages) and `homepage`/`subpage` likely account for the majority of clicks.
> - The **gap between completers and non-completers** varies by activity type. Some resource types (e.g., forums, quizzes) may show a proportionally larger gap, suggesting they are more "discriminating" engagement signals.
> - Activity type **diversity** shows a positive association with completion: students who use more resource types tend to complete at higher rates. This is another dose-response pattern, complementing the decile analysis in Section 4.
>
> **Caveat:** Activity type diversity is partly confounded with total engagement volume — students who click more are more likely to encounter different resource types. The diversity signal may not be independent of the volume signal. BQ2 will test this more formally.
>
> **SaaS parallel:** Feature breadth (number of distinct features used) is a common retention predictor in product analytics. Users who explore beyond the core feature set tend to be more engaged and less likely to churn.

## 8. Course-Level Engagement Profiles

Notebook 01 (Section 7) examined course *design* characteristics — assessment density, VLE resource count, course length. Here we add the **engagement dimension**: how much do students actually click in each course?

This matters because courses with more resources may naturally generate more clicks. An engagement level that is "low" in a resource-heavy course might be "normal" in a lighter one. BQ4 (Notebook 06) will formalize this analysis; here we establish the baseline.

In [ ]:
# --- Course-level engagement summary ---
df_course_engagement = execute_query('''
    SELECT
        ee.code_module,
        COUNT(*) AS n_active_enrollments,
        ROUND(AVG(ee.total_clicks_first_28), 0) AS mean_clicks,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.total_clicks_first_28), 0
        ) AS median_clicks,
        ROUND(AVG(ee.active_days_first_28), 1) AS mean_active_days,
        ROUND(
            PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY ee.active_days_first_28), 0
        ) AS median_active_days,
        ROUND(AVG(ee.avg_clicks_per_active_day), 1) AS mean_intensity
    FROM v_engagement_early ee
    GROUP BY ee.code_module
    ORDER BY median_clicks DESC
''')

print('=== Course-Level Engagement Summary (first 28 days) ===\n')
df_course_engagement

In [ ]:
# --- Box plots of total clicks by module ---
# Shows within-course engagement variation: some courses may have tight
# distributions (all students click similarly) while others have wide spread.
df_course_clicks = execute_query('''
    SELECT
        ee.code_module,
        ee.total_clicks_first_28
    FROM v_engagement_early ee
''')

# Cap at 99th percentile to manage visual outliers
p99_course = df_course_clicks['total_clicks_first_28'].quantile(0.99)
df_course_clicks['clicks_capped'] = df_course_clicks['total_clicks_first_28'].clip(
    upper=p99_course,
)

# Order modules by median clicks for consistent reading
module_order = (
    df_course_clicks.groupby('code_module')['total_clicks_first_28']
    .median()
    .sort_values(ascending=False)
    .index.tolist()
)

fig, ax = plt.subplots(figsize=FIG_SIZE)
sns.boxplot(
    data=df_course_clicks, x='code_module', y='clicks_capped',
    order=module_order, color='#4C72B0', ax=ax,
)
ax.set_xlabel('Module')
ax.set_ylabel(f'Total clicks (first 28 days, capped at p99 = {p99_course:,.0f})')
ax.set_title('Engagement Distribution by Module')
sns.despine()
fig.tight_layout()
save_fig(fig, '02_course_engagement_boxplot')
plt.show()

> **Interpretation:**
> - Engagement levels vary **substantially** across modules. Some courses generate much higher click volumes than others, likely reflecting differences in course design (number of VLE resources, assessment structure, content format).
> - Within each course, there is also considerable variation — the box plot whiskers and outliers show that some students click many times more than their peers in the same course.
> - The mean-median gap in the summary table confirms right-skewed distributions within most courses: a few highly active students pull the mean above the median.
>
> **Implication:** Engagement metrics should be interpreted **relative to the course**, not in absolute terms. This is why the engagement decile (Section 4) normalizes within each course-presentation. A student with 100 clicks may be in the top decile of one course and the bottom decile of another. BQ4 (Notebook 06) will analyze how course design features relate to engagement levels and retention.

## 9. Engagement-Outcome Correlation Matrix

We now synthesize all early engagement metrics into a single view: which metrics have the **strongest linear association** with completion?

We compute Pearson correlations between all engagement variables and the binary `completed` variable. When one variable is binary (0/1) and the other is continuous, the Pearson correlation equals the point-biserial correlation — a standard measure of association between a continuous and a dichotomous variable.

**Important:** This analysis uses a `LEFT JOIN` to include ghost students (with engagement values set to 0). This gives the full-population perspective: ghost students are the extreme low end of engagement, and including them strengthens correlations because they almost all fail to complete.

In [ ]:
# --- Full engagement + outcome data for correlation analysis ---
# LEFT JOIN includes ghost students (COALESCE to 0) for the full population view.
# Using -1 for last_active_day when NULL to indicate "never active",
# keeping it numerically distinct from day 0.
df_corr_data = execute_query('''
    SELECT
        COALESCE(ee.active_days_first_28, 0)          AS active_days,
        COALESCE(ee.total_clicks_first_28, 0)         AS total_clicks,
        COALESCE(ee.avg_clicks_per_active_day, 0)     AS clicks_per_day,
        COALESCE(ee.last_active_day_in_window, -1)    AS last_active_day,
        COALESCE(ee.engagement_decile_in_course, 0)   AS engagement_decile,
        se.completed
    FROM v_student_enriched se
    LEFT JOIN v_engagement_early ee
        ON se.id_student = ee.id_student
        AND se.code_module = ee.code_module
        AND se.code_presentation = ee.code_presentation
''')

# Shorter column labels for readability in the heatmap
label_map = {
    'active_days': 'Active days',
    'total_clicks': 'Total clicks',
    'clicks_per_day': 'Clicks/day',
    'last_active_day': 'Last active day',
    'engagement_decile': 'Engagement decile',
    'completed': 'Completed',
}
df_corr_renamed = df_corr_data.rename(columns=label_map)
corr_matrix = df_corr_renamed.corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap=PALETTE_SEQUENTIAL,
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    ax=ax,
)
ax.set_title('Engagement-Outcome Correlation Matrix\n(includes ghost students as zeros)')
fig.tight_layout()
save_fig(fig, '02_engagement_correlation_matrix')
plt.show()

In [ ]:
# --- Ranked correlations with 'completed' ---
# Extract the correlation of each engagement metric with the binary outcome
# and rank by absolute strength.
corr_with_completed = (
    corr_matrix['Completed']
    .drop('Completed')
    .reset_index()
)
corr_with_completed.columns = ['metric', 'correlation']
corr_with_completed['abs_correlation'] = corr_with_completed['correlation'].abs()
corr_with_completed = corr_with_completed.sort_values('abs_correlation', ascending=False)
corr_with_completed = corr_with_completed.drop(columns='abs_correlation')

print('=== Engagement Metrics Ranked by Correlation with Completion ===\n')
print(corr_with_completed.to_string(index=False))

> **Interpretation:**
> - The engagement metrics are **inter-correlated** — active days, total clicks, and engagement decile tend to move together. This is expected: students who click more also tend to click on more days.
> - The strongest correlates with completion are likely **active days** and **total clicks** — the simplest engagement measures are also the most predictive.
> - **Clicks per active day** (intensity) may have a weaker correlation with completion than active days (frequency), reinforcing the Section 6 finding that consistency matters more than intensity.
> - Including ghost students (as zeros) strengthens all correlations because they represent the extreme case: zero engagement, near-zero completion.
>
> **Caveat:** Pearson correlations capture **linear** relationships. Non-linear patterns (e.g., diminishing returns at high engagement levels, or a threshold effect) would be underestimated. The decile analysis in Section 4 partially addresses this by looking at the shape of the dose-response curve.
>
> **Looking ahead:** BQ2 (Notebook 04) will formalize these associations with t-tests, effect sizes (Cohen's d), and confidence intervals — moving from correlation to actionable signal assessment.

## 10. Key Takeaways and Next Steps

### What we learned

1. **Ghost students are the clearest signal.** A measurable share of enrollments have zero VLE activity in the first 28 days. Their completion rate is near zero. They are the lowest-hanging fruit for intervention — they need *activation*, not academic support.

2. **Engagement separates outcomes across all metrics.** Completers show substantially higher active days, total clicks, and later last-active-day in the 28-day window. The separation is visible in every distribution plot.

3. **Dose-response relationship.** Completion rate increases monotonically with engagement decile. The bottom 2–3 deciles are extreme-risk segments with far-below-average completion rates.

4. **Early divergence.** The daily trajectory shows that the engagement gap between completers and non-completers opens within the first 7–10 days. Non-completers both click less *and* stop showing up earlier — a dual signal of intensity and frequency.

5. **Consistency over intensity.** The typology analysis reveals that *frequency* (number of active days) predicts completion more strongly than *intensity* (clicks per active day). Steady engagement outperforms binge engagement.

6. **Activity type diversity matters.** Students who use more resource types complete at higher rates. Breadth of engagement is a positive signal, though partly confounded with total volume.

7. **Course-level variation.** Engagement levels differ substantially across modules, reinforcing the need for course-aware analysis. Within-course normalization (engagement deciles) addresses this.

8. **Strongest correlates.** Active days and total clicks have the highest correlation with completion, followed by engagement decile and last active day. These are candidates for an early warning system.

### What comes next

| Notebook | Business Question | Focus |
|----------|------------------|-------|
| **03** | BQ1 | Where and when do students drop out? |
| **04** | BQ2 | Which early behavioral signals predict drop-out? |
| **05** | BQ3 | Demographics vs. behavior — what predicts outcome more? |
| **06** | BQ4 | How do course characteristics affect retention? |
| **07** | BQ5 | Top 3 actionable interventions |

---

**Reproducibility:** All figures are saved to `reports/figures/`. To reproduce this notebook, run `python -m run_pipeline` first, then execute all cells in order.

> **From EDA to analysis:** These two EDA notebooks (01 + 02) have established the population profile, engagement landscape, and key hypotheses. From here, the remaining notebooks answer specific business questions with statistical rigor.
>
> Continue to **Notebook 03** (`03_bq1_dropout_timing.ipynb`) for BQ1: where and when do students drop out?